# Week 1 — Environment, Florence-2 Zero-Shot Baseline

CM4610 Part B — VLM-Based GUI Grounding Agent

Deliverables this week: inference script + baseline table (per brief Section 10).

This notebook calls the scripts in `scripts/` and `src/` directly rather than duplicating logic, so the notebook stays the narrative/results layer and the code stays testable and reusable.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

## 1. Environment check

Run this once per machine setup. Fix any `[FAIL]` before continuing.

In [ ]:
!python ../scripts/check_env.py

## 2. Download + normalize ScreenSpot subset

**Action item:** confirm the HF repo id in `configs/config.yaml` (`data.screenspot.hf_repo`) is current before running this — see note in `src/data/screenspot_loader.py`.

In [ ]:
from src.data.screenspot_loader import load_config, download_screenspot, normalize_screenspot_split, save_examples_jsonl

cfg = load_config('../configs/config.yaml')
cfg['data']['screenspot']

In [ ]:
# Uncomment to actually download (can be slow on first run):
# ss_cfg = cfg['data']['screenspot']
# ds = download_screenspot('../' + ss_cfg['cache_dir'], ss_cfg['hf_repo'])
# split_name = 'test' if 'test' in ds else list(ds.keys())[0]
# examples = normalize_screenspot_split(ds[split_name], image_out_dir='../' + ss_cfg['cache_dir'] + '/images', limit=100)
# save_examples_jsonl(examples, '../' + ss_cfg['cache_dir'] + '/screenspot_normalized.jsonl')
# len(examples)

## 3. Sanity-check Florence-2 zero-shot on a single example

**Action item:** confirm `<CAPTION_TO_PHRASE_GROUNDING>` is the right task prompt against the Florence-2 model card before trusting results below.

In [ ]:
from src.inference.florence2_inference import Florence2Grounder, draw_prediction

grounder = Florence2Grounder(
    model_id=cfg['model']['base_model_id'],
    task_prompt=cfg['model'].get('task_prompt', '<CAPTION_TO_PHRASE_GROUNDING>'),
)

In [ ]:
# image_path = '../data/screenspot/images/screenspot_000000.png'
# image = Image.open(image_path).convert('RGB')
# instruction = 'click the search bar'
# result = grounder.ground(image, instruction)
# print(result.bbox_xyxy, f"{result.latency_ms:.1f} ms")
# draw_prediction(image, result.bbox_xyxy, '../results/sanity_check.png')
# Image.open('../results/sanity_check.png')

## 4. Zero-shot baseline table

This is the Week 1 deliverable: click accuracy, mean IoU and latency on a ScreenSpot subset, overall and per category. Every later result (fine-tuned, fine-tuned + classical CV aid) will be compared against this.

In [ ]:
!python ../scripts/run_zeroshot_baseline.py --n 50 --out ../results/week1_zeroshot_baseline.csv

In [ ]:
df = pd.read_csv('../results/week1_zeroshot_baseline.csv')
df.head()

In [ ]:
if df['category'].notna().any():
    summary = df.groupby('category')[['click_correct', 'iou']].mean()
    summary.plot(kind='bar', figsize=(8, 4), title='Zero-shot Florence-2: accuracy & IoU by category')
    plt.ylabel('score')
    plt.tight_layout()
    plt.savefig('../results/week1_category_breakdown.png')
    plt.show()

## 5. Notes / next steps for Week 2

- [ ] Confirm real deadline on Moodle
- [ ] Confirm ScreenSpot HF repo id
- [ ] Confirm Florence-2 task prompt
- [ ] Begin self-collected screenshot capture + annotation (target 150-300, split rationale documented)
- [ ] Log every GenAI-assisted step in `docs/genai_usage_log.md`